# Inventory Decision Intelligence — real-data retail planning

**Business assignment:** turn retail transactions into feasible replenishment decisions under uncertain demand, limited cash and warehouse capacity.

This is the complete project workflow: data acquisition → cleaning → SQL → segmentation → forecasting → uncertainty → network simulation → optimization → test results → planner → export.

**Data:** [UCI Online Retail II](https://doi.org/10.24432/C5CG6D), contributed by Daqing Chen, CC BY 4.0. About 1.07 million original transaction rows. Real sales and selling prices; proposed network and operating assumptions. Modeling excludes Customer IDs; extended EDA inspects only their aggregate completeness and exports no identifiers.

Start with `PROFILE='quick'`; use `'full'` for 150 products and six training-selected markets. Both are complete pipelines. The real retailer's actual inventory and stockouts are unknown. Reported inventory outcomes are model results, not verified company savings.

## 1. Reproducible setup

The notebook uses the repository's tested modules. The source commit is printed so you can repeat the exact version. Set `REVISION` to a recorded commit for an exact source checkout. The first run downloads a 45 MB ZIP and reads the Excel workbook; subsequent runs reuse the cache.

In [ ]:
import sys,subprocess,os,time,json,hashlib,zipfile
from pathlib import Path
from datetime import datetime,timezone
REPOSITORY='https://github.com/buckminster07/multi-echelon-inventory-optimization.git'
REVISION='main'
if globals().get('LOCAL_REPO'):
    repo=Path(LOCAL_REPO).resolve()
else:
    repo=Path('/content/inventory-decision-intelligence')
    if not repo.exists():subprocess.check_call(['git','clone',REPOSITORY,str(repo)])
    subprocess.check_call(['git','-C',str(repo),'fetch','origin',REVISION])
    subprocess.check_call(['git','-C',str(repo),'checkout','--detach','FETCH_HEAD'])
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(repo/'retail_requirements.txt')])
os.chdir(repo);sys.path.insert(0,str(repo))
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:pass
import numpy as np,pandas as pd
from IPython.display import display,Image,HTML
from retail_planner.config import Config
PROFILE=globals().get('PROFILE','quick')
c=Config() if PROFILE=='full' else Config.quick()
started=time.perf_counter()
out=Path('outputs')/('colab-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
out.mkdir(parents=True,exist_ok=False)
display(pd.Series(c.dict()))

## 2.1 First, look at the original dataset

Run this cell before cleaning anything. Each row is a product line on an invoice, not an entire customer order. The preview shows original values with source column names. Customer IDs are omitted from the preview and modeling; only their aggregate missing-value counts are retained. Loading both Excel sheets can take a few minutes.

**Look for:** the product, units, selling price, transaction time and destination country. A country is a sales market, not an observed warehouse.

In [ ]:
from retail_planner.data import acquire,clean_transactions,prepare,sql_store
from retail_planner.eda import load_raw,sale_mask,product_mask,seasonality,price_changes,return_rates,cutoff_sensitivity,plot_eda_section,export_eda
archive=acquire('data/cache')
raw,missing=load_raw(archive,'data/cache')
source_names={'invoice':'Invoice','sku':'StockCode','description':'Description','quantity':'Quantity','timestamp':'InvoiceDate','price':'Price','market':'Country'}
print(f'Original rows: {len(raw):,}; original workbook columns: {missing.loc[missing.sheet.eq("ALL")].shape[0]}')
print('First 10 original transaction lines (before cleaning):')
display(raw[list(source_names)].head(10).rename(columns=source_names))

## 2.2 Understand the columns and basic ranges

Inspect data types, coverage, distinct products and markets, and numeric ranges. The loader standardizes identifier strings and adds a parsed `date`, but has not removed transaction rows. Negative quantities and zero prices should prompt investigation, not immediate averaging into demand.

In [ ]:
display(pd.DataFrame({'field':list(source_names.values()),'meaning':['Invoice identifier','Product/SKU identifier','Product description','Units on this invoice line','Transaction timestamp','Unit selling price in GBP','Destination market']}))
display(raw[list(source_names)].dtypes.astype(str).rename('loaded_dtype').to_frame())
display(pd.Series({'first_date':raw.date.min(),'last_date':raw.date.max(),'distinct_product_codes':raw.sku.nunique(),'distinct_markets':raw.market.nunique(),'rows':len(raw)}))
display(raw[['quantity','price']].describe(percentiles=[.01,.5,.95,.99]))

## 2.3 Check missing values, then draw the chart

Counts come from every original workbook column, before cleaning, for each sheet and overall. Missing means null or blank text. Customer-ID missingness matters for customer analytics, but the inventory pipeline uses product and market identifiers.

**Read the chart:** bar length is the percentage missing, not the number of rows removed. A present value can still be invalid.

In [ ]:
eda_out=out/'eda'
eda_tables={'missing_values':missing}
display(missing.query("sheet == 'ALL'")[['field','rows','missing','missing_pct']])
display(missing.query("sheet != 'ALL'")[['sheet','field','missing','missing_pct']])
display(Image(filename=str(plot_eda_section('missing_values',eda_tables,eda_out))))

## 2.4 Inspect suspicious records before cleaning

Separate cancellations, negative quantities, nonpositive prices and nonproduct codes. These flags overlap. Repeated lines are counted but retained because no invoice-line ID proves an accidental duplicate. Examples below are restricted to the training period; later dates remain reserved for evaluation.

In [ ]:
training_raw=raw.loc[raw.date<=c.train_end].copy()
flags=pd.Series({
    'cancel_invoice_prefix':training_raw.invoice.str.upper().str.startswith('C').fillna(False).sum(),
    'negative_quantity':(training_raw.quantity<0).sum(),
    'zero_quantity':(training_raw.quantity==0).sum(),
    'nonpositive_price':(training_raw.price<=0).sum(),
    'nonproduct_code':(~product_mask(training_raw)).sum(),
    'repeated_loaded_lines':training_raw.duplicated().sum(),
    'quantity_above_10000':(training_raw.quantity>10000).sum()})
display(flags.rename('training_rows_flagged').to_frame())
display(training_raw.loc[training_raw.quantity<0,list(source_names)].head(5))
display(training_raw.loc[training_raw.quantity>10000,list(source_names)].head(5))

## 2.5 Clean the sales data and compare before/after

Positive outbound sales are the demand proxy. Exclude cancellations, invalid dates, nonproduct codes, nonpositive quantities/prices and lines above 10,000 units. Keep repeated lines. Returns are analyzed separately, not subtracted from future demand. The audit below covers the whole workbook; the next analysis cells use only training dates.

**Check:** the number of remaining rows, the first cleaned records, and the documented reasons for removal.

In [ ]:
transactions,audit=clean_transactions(archive,'data/cache')
display(pd.DataFrame({'stage':['Original','Cleaned'],'rows':[len(raw),len(transactions)]}))
display(pd.Series(audit))
display(transactions[['date','sku','description','quantity','price','market','revenue']].head(10))
eda_sales=training_raw.loc[sale_mask(training_raw)&(training_raw.quantity<=10000)].copy()
print('Behavioral EDA ends at:',c.train_end,'; training sales lines:',len(eda_sales))

## 2.6 Plot daily sales and the underlying trend

Aggregate units by calendar day and include days with zero recorded sales. The 28-day centered average is a descriptive smoother, not a forecast feature. This analysis covers all eligible training products, not just the later selected cohort.

**Read the chart:** isolated spikes may be bulk orders; a sustained rise may reflect trend or seasonality. Zero sales do not prove zero latent demand.

In [ ]:
import matplotlib.pyplot as plt
daily,weekly,monthly,seasonality_evidence=seasonality(eda_sales,c.train_end)
eda_tables.update(seasonal_daily=daily,weekday_profile=weekly,monthly_profile=monthly)
display(daily[['date','units','trend_28d']].head(10))
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(daily.date,daily.units,alpha=.45,label='Daily units')
ax.plot(daily.date,daily.trend_28d,color='#a84931',label='28-day centered mean')
ax.set(title='Training-period sales and trend',xlabel='Date',ylabel='Units');ax.legend()
fig.tight_layout();fig.savefig(eda_out/'daily_trend.png',dpi=140)
display(Image(filename=str(eda_out/'daily_trend.png')));plt.close(fig)

## 2.7 Examine weekday seasonality

Compare sales relative to their local trend across weekdays. The Friedman test compares weekdays within complete weeks; Kendall's W summarizes effect size. Serial dependence and business opening schedules limit interpretation of the nominal p-value.

**Read the chart:** 1 means sales equal the local daily trend. A weekday effect does not prove that consumer preferences caused it.

In [ ]:
display(weekly)
display(pd.Series(seasonality_evidence))
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],weekly.mean_relative_units,color='#277f8e')
ax.axhline(1,color='gray',linestyle='--')
ax.set(title='Weekday sales relative to local trend',ylabel='Sales / 28-day trend',xlabel='Weekday')
fig.tight_layout();fig.savefig(eda_out/'weekday_pattern.png',dpi=140)
display(Image(filename=str(eda_out/'weekday_pattern.png')));plt.close(fig)

## 2.8 Examine monthly patterns

Compare average units per calendar day, rather than unequal monthly totals. Only complete months enter the plot. The training data contains fewer than two full annual cycles, so this chart alone cannot establish repeatable annual seasonality.

**Look for:** the months with the largest mean daily sales, then distinguish a descriptive peak from evidence that it will repeat.

In [ ]:
display(monthly)
complete_months=monthly.loc[monthly.complete_month]
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(complete_months.month,complete_months.mean_daily_units,color='#277f8e')
ax.tick_params(axis='x',rotation=60)
ax.set(title='Monthly sales patterns: complete training months',ylabel='Mean units per calendar day',xlabel='Month')
fig.tight_layout();fig.savefig(eda_out/'monthly_pattern.png',dpi=140)
display(Image(filename=str(eda_out/'monthly_pattern.png')));plt.close(fig)

## 2.9 Analyze selling-price changes

Calculate the median selling price within each product, market and month. Compare consecutive months only when each has at least five sales lines. This controls product/market composition, but not customer or bulk-discount mix.

**Read the plots:** the histogram shows the distribution of monthly changes; the lines index example prices to 100 in their first qualifying month. These are not estimates of price elasticity.

In [ ]:
prices=price_changes(eda_sales)
eda_tables['monthly_prices']=prices
display(prices.loc[prices.change_pct.notna()].head(10))
display(prices.change_pct.dropna().describe())
print('Comparisons changing by more than 1%:',f'{100*prices.change_pct.dropna().abs().gt(1).mean():.2f}%')
display(Image(filename=str(plot_eda_section('price_changes',eda_tables,eda_out))))

## 2.10 Compare market return ratios

Use the training records before the bulk cutoff. Divide negative-quantity product units by gross positive outbound units in each market. Returns are not linked to their original purchases; this is not the probability of returning an order.

**Read the plots:** inspect both the ratio and its sales-volume denominator. Investigate large cancellations before attributing a high ratio to market quality.

In [ ]:
returns=return_rates(training_raw)
eda_tables['market_returns']=returns
display(returns.head(12))
display(Image(filename=str(plot_eda_section('market_returns',eda_tables,eda_out))))

## 2.11 Test the extreme-order cutoff

Compare maximum line quantities of 1,000, 5,000, 10,000 and no cutoff using the same training history. Measure how much recorded volume and revenue each rule retains and how it changes the biggest daily spike.

**Read the plots:** removing few rows can still remove substantial units. This is data sensitivity, not a rerun of the inventory policies; the existing 10,000-unit rule remains fixed.

In [ ]:
cutoffs=cutoff_sensitivity(training_raw,c.train_end)
eda_tables['cutoff_sensitivity']=cutoffs
display(cutoffs)
display(Image(filename=str(plot_eda_section('cutoff_sensitivity',eda_tables,eda_out))))

## 2.12 Save the EDA and connect it to planning

Export the tables and figures already calculated above. No second raw-data read or analysis run is required. Before continuing, explain one finding about data quality, one about demand, and one limitation. Next, select the modeling cohort and build the SQL tables.

In [ ]:
export_eda(eda_tables,seasonality_evidence,archive,eda_out,c.train_end)
print('Saved EDA report:',eda_out/'README.md')
display(pd.DataFrame({'finding':['Missing customer IDs','Intermittent/variable demand','Weekday effects','Changing transaction prices','Extreme quantities'],
    'planning_implication':['Use SKU/market aggregation; avoid unsupported customer conclusions','Compare intermittent methods and uncertainty buffers','Test calendar-aware forecasts; inspect opening schedules','Separate selling-price evidence from assumed acquisition costs','Document cutoff sensitivity before interpreting shortages']}))
del raw,training_raw,eda_sales

## 3. Select the cohort and integrate through SQL

Markets and products are ranked using only data through April 2011. The initial assortment is fixed before later evaluation. A zero-filled active pair means zero observed sales, not confirmed zero latent demand. SQLite keys separate products, locations and active pairs; the modeling array is reconstructed through SQL.

**Learn:** a country is a demand market; its proposed fulfillment hub is an operating assumption. **Exercise:** why would choosing the highest-selling products across the entire dataset leak future information?

In [ ]:
data=prepare(transactions,c)
db=sql_store(data,c,out/'planning.sqlite')
print('Markets:',data['markets'])
print('Products:',len(data['skus']),'Active pairs:',int(data['active'].sum()))
display(pd.read_sql_query("SELECT location,COUNT(*) AS stocked_products FROM assortment GROUP BY location",db))
display(pd.read_sql_query("SELECT a.location,a.sku,SUM(d.units) AS units FROM assortment a JOIN daily_sales d USING(location,sku) WHERE d.date<=? GROUP BY a.location,a.sku ORDER BY units DESC LIMIT 10",db,params=(c.train_end,)))
assert not db.execute('PRAGMA foreign_key_check').fetchall()
data['sku_master'].to_csv(out/'sku_master.csv',index=False)
data['pairs'].to_csv(out/'assortment.csv',index=False)

## 4. Demand diagnostics and segmentation

ABC describes training sales-value concentration. CV is standard deviation divided by mean; XYZ segments that variability. ADI is elapsed days divided by nonzero-demand days. These descriptive segments do not automatically determine an optimal policy.

**Exercise:** compare a high-volume UK pair with an intermittent export pair. Explain why portfolio WAPE can hide poor small-market performance.

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,3.6))
axes[0].plot(data['dates'],data['y'].sum(axis=(1,2)));axes[0].set(title='Observed sales for the fixed cohort',ylabel='Units')
axes[1].hist(data['pairs'].query('active').zero_fraction,bins=15);axes[1].set(title='Zero-sales frequency across active pairs',xlabel='Fraction of days')
fig.tight_layout();plt.show()
display(data['sku_master'].head(12));display(data['pairs'].query('active').sort_values('adi',ascending=False).head(10))
del transactions

## 5. Train and select forecasts

Compare seasonal naive, a 28-day mean, TSB for intermittent demand, and a global Poisson gradient booster. The booster learns across product–market pairs using categorical IDs, calendar fields and lagged demand statistics. Its recursive forecast uses predictions as future lags. May backtests select a method separately for each pair.

**Math:** TSB forecast = occurrence probability × positive-demand size. **Exercise:** if probability is 0.2 and positive size is ten, what is expected daily quantity?

In [ ]:
from retail_planner.forecast import train_booster,select_models,calibrate,forecast_window,end_index,summarize_forecasts,MODELS
model,training_rows=train_booster(data,c)
choice,selection=select_models(data,c,model)
display(selection.groupby('model')[['mae','rmse','bias','wape']].mean())
display(pd.Series([MODELS[i] for i in choice[data['active']]]).value_counts().rename('selected_pairs'))
print('Global-model training rows:',training_rows)

## 6. Calibrate cumulative uncertainty

June–July data calibrates cumulative forecast errors. Demand over a lead/review interval is a sum, so its uncertainty is different from one-day error. The 5th and 95th empirical error quantiles form a 90% interval. Simultaneous errors across markets are aggregated for the DC buffer.

**Exercise:** explain why summing six separate 95th percentiles usually does not give the 95th percentile of total demand. Inspect held-out coverage later; calibration is not a guarantee under distribution shift.

In [ ]:
cal=calibrate(data,c,model,choice)
display(pd.DataFrame([dict(horizon=h,calibration_origins=len(v['errors']),average_upper_buffer=v['buffer'][data['active']].mean()) for h,v in cal.items()]))
windows={};forecast_rows=[];coverage_rows=[]
for name,left,right in [('tuning',c.calibration_end,c.tuning_end),('validation',c.tuning_end,c.validation_end),('test',c.validation_end,c.test_end)]:
    a=end_index(data,left);b=end_index(data,right)
    f,m,cov=forecast_window(data,c,model,choice,a,b,cal)
    windows[name]=(f,data['y'][a:b]);m['split']=name;cov['split']=name
    forecast_rows.append(m);coverage_rows.append(cov)
forecast_errors=pd.concat(forecast_rows,ignore_index=True)
coverage=pd.concat(coverage_rows,ignore_index=True)
print('Test forecast diagnostics are displayed after policy choices are frozen below.')

## 7. Define policies and run an accounting check

Hub IP = on-hand + shipped inbound + queued requests. Compare fixed cover, daily (r,Q), weekly order-up-to and forecast-aware targets. Every request is constrained by packs, MOQ, item space, total space and rolling seven-day purchasing budget.

**Example:** ten units/day × ten protection days + 25 buffer gives target 125. IP 77 leaves a 48-unit gap, or eight six-unit packs. Limited shared budget may reduce that order.

The simulator receives supply, serves customers, records loss, reviews requests, allocates DC stock, places supplier orders and charges costs in that order.

In [ ]:
from retail_planner.simulation import simulate,POLICIES,SCENARIOS
check=simulate(data,c,*windows['tuning'],cal,'periodic','normal',c.seed+900,record=True)
display(pd.Series(check[0]));display(check[1].head());display(check[3].head())
assert check[0]['fulfilled_units']+check[0]['lost_units']==check[0]['demand_units']
assert (check[3].due>check[3].placed).all()
print('Daily conservation, pipeline, budget, pack and capacity checks are enforced inside the engine.')

## 8. Optimize joint inventory targets

Eight hub/DC multiplier pairs are evaluated on August normal, spike and delay scenarios. Combined stress is excluded. First prefer candidates meeting 95% aggregate fill and 97% A-item fill in every tuning scenario; among them minimize cost. If no tested candidate qualifies, select the least service violation and report that fact.

**Learn:** finite-grid optimization does not prove a global optimum. **Exercise:** explain why the cheapest unconstrained policy may be unacceptable to the business.

In [ ]:
from retail_planner.evaluation import tune,evaluate,summarize,sensitivity
best,search=tune(data,c,*windows['tuning'],cal)
display(search.sort_values(['service_gap','mean_cost']))
print('Frozen settings:',best)
print('Any service-feasible candidate:',bool(search.service_feasible.any()))

## 9. Validate and evaluate on untouched test months

September is diagnostic validation; October–November is the final test. All policies share the same demand and lead realizations for a seed. Paired cost differences compare them within seed. The intervals are conditional on observed sales and scenario assumptions, not uncertainty about all future markets.

**Exercise:** find a scenario where a policy improves service but raises cost. Explain the business choice rather than selecting one headline percentage.

In [ ]:
validation=evaluate(data,c,*windows['validation'],cal,best,seed_offset=3000,replications=5)
raw=evaluate(data,c,*windows['test'],cal,best)
summary,paired=summarize(raw)
display(summary);display(paired)
forecast_summary=summarize_forecasts(forecast_errors.query("split=='test'"))
display(forecast_summary)
display(coverage.query("split=='test'").groupby('horizon')[['covered','width']].mean())

## 10. Sensitivity and operational drill-down

Vary budget, annual holding rate and unobserved-demand uplift with the chosen policy frozen. Cash budgets constrain decisions; lost-margin and holding assumptions affect tradeoffs. Historical sales may understate latent demand, so demand uplift is a stress bound rather than recovered truth.

**Exercise:** inspect an A-item with low service. Is the cause forecast error, upstream availability, MOQ, budget or capacity? Use shipments and proposal records to investigate.

In [ ]:
sens=sensitivity(data,c,*windows['test'],cal,best)
display(sens.groupby(['parameter','value'])[['relevant_cost_gbp','fill_rate','lost_units']].mean())
example=simulate(data,c,*windows['test'],cal,'forecast_optimized','normal',c.seed+5000,record=True,**best)
display(example[4].sort_values('fill_rate').head(12))

## 11. Planner workflow and decision replay

Choose a product/location/policy/scenario, review the proposals, and approve, reduce or reject a quantity. A planner name and reason are required. Quantity increases need a new batch allocation because other products already reserve cash and capacity. Duplicate decisions are rejected transactionally.

**Evaluate decisions** reruns the same case with selected order quantities capped by your recorded decisions. Other orders remain automated. The audit is real SQLite persistence; supplier dispatch is simulated.

In [ ]:
from retail_planner.planner import interface,save_proposals
save_proposals(db,f'planner-forecast_optimized-normal-{c.seed+5000}','forecast_optimized','normal',c.seed+5000,example[2])
planner_state=interface(data,c,*windows['test'],cal,best,raw,db)

## 12. Export results, audit and reproduction evidence

The export includes the database, individual trials, paired comparisons, forecasts, coverage, SKU-market service, proposals, shipments, assumptions, source/data checksums and a readable report. Rerun this cell after making additional planner decisions. Colab storage is temporary: download the ZIP.

**Explain the project:** use the business brief, methods and learning path in the repository. Clearly distinguish observed sales from assumed operations and modeled business outcomes.

In [ ]:
from retail_planner.pipeline import finalize
frames={'forecast_selection':selection,'forecast_errors':forecast_errors,'interval_coverage':coverage,'forecast_summary':forecast_summary,'tuning':search,'validation_trials':validation,'trials':raw,'policy_summary':summary,'paired_differences':paired,'sensitivity':sens,'daily_trace':example[1],'proposals':example[2],'shipments':example[3],'pair_service':example[4]}
db.commit()
pd.read_sql_query('SELECT * FROM decision',db).to_csv(out/'planner_decisions.csv',index=False)
manifest=finalize(data,c,best,frames,audit,training_rows,out,started)
display(Image(filename=str(out/'policy_comparison.png')))
import base64
report_html=(out/'report.html').read_text(encoding='utf-8')
for chart_path in out.rglob('*.png'):
    encoded=base64.b64encode(chart_path.read_bytes()).decode('ascii')
    report_html=report_html.replace('src="'+chart_path.relative_to(out).as_posix()+'"','src="data:image/png;base64,'+encoded+'"')
display(HTML(report_html))
archive_path=out.with_suffix('.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(out))
print('Export:',archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:print('Local run: open the output ZIP directly.')